In [2]:
####################################
#ENVIRONMENT SETUP

In [3]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd
import metpy.calc as mpcalc

#data classes
import xarray as xr
import pickle

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [4]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [5]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RadarData/RadarObservationMask"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

 DirectoryManager_Class Summary
 Main Directory:           /glade/u/home/aroseman/Projects/Regional-MPAS-Project
 Main Scratch Directory:   /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project
 Scratch Directory:        /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0
 Data Directory:           /glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA
 Main Output Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT
 Main Output Plotting Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/PLOTTING
 Main Code Directory:      /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles
 Current Code Directory:   /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/DataAnalysis/Observation_Data/TRACER&Hawaii/RadarData



In [6]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [7]:
# spinup_hours = "0"
# RunType = ("TRACER","WET","NSSL",spinup_hours)
# spinup_hours = "-5"
# RunType = ("TRACER","DIURNAL","NSSL",spinup_hours)
spinup_hours = "0"
RunType = ("Hawaii","TRADES","NSSL",spinup_hours)

ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

Found 193/193 files matching expected times.
Opened history file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_NSSL/model_run_spinup0hrs/history_cartesian/history.2022-08-08_00.00.00.latlon.nc
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_NSSL/model_run_spinup0hrs/diag_cartesian/diag.2022-08-08_00.00.00.latlon.nc

=== MPAS Structured (lat-lon) Model Data Summary ===
 Region:         Hawaii
 Case:           TRADES
 Microphysics:   NSSL
 Resolution:     20-1km
 Time Step:      15mins
 Time Range:     2022-08-08 to 2022-08-10
 Coordinates:    ['latitude', 'longitude', 'nVertLevels', 'nVertLevelsP1']
 # History Files:193
 # Diag Files:   193
 # Time Steps:   193
 Data Directory: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_NSSL/model_run_spinup0hrs
 Static File:    Hawaii_regional2

In [8]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [9]:
#Importing ERA5 Data Loading Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","ERA5_Data"))
from CLASSES_ERA5DataLoading import ERA5DataLoading_Class,ERA5DataLoading_Class_gdex

In [10]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [11]:
########################
#DATA INFORMATION

In [12]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. X-Band Scanning ARM Cloud Radar (XSACRCFRQC), 2022-06-09 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by Y. Feng, A. Matthews, E. Schuman, K. Johnson, I. Lindenmaier, V. Castro and T. Wendler. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/2001296.

#Globus Download Link
# https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261781*__;Ly8v!!PvDODwlR4mBZyAb0!REKAOzHJNvJk50CY5Pjl135CV83BhArwtdyMDuBM-28KqreBug8Xb5Mc3MLgy_p9PUiWe2uVXq-EfUtLcGdH5w$

In [13]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. Ka-Band Scanning ARM Cloud Radar (KASACRCFRQC), 2022-06-08 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by I. Lindenmaier, K. Johnson, D. Nelson, A. Matthews, T. Wendler, V. Melo de Castro, M. Rocque and Y. Feng. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/1877338.

# https://armgov.svcs.arm.gov/capabilities/instruments/kasacr

#Globus Download Link
#https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261893*__;Ly8v!!PvDODwlR4mBZyAb0!UJrKWg_xaYuaaa_iaY91TCVMB_neSskXsDKHtPPCG7Ix6sEmiEvnngTUrYuV18LudZqxB3eHyTDuCHSZ3o3zrg$

In [14]:
#LOADING RADAR CLASS
if int(ModelData_NSSL.spinup_hours) <= 0:
    if ModelData_NSSL.region == "TRACER":
        if ModelData_NSSL.case == "WET":
            dateString = '2022-06-30_2022-07-03'
        elif ModelData_NSSL.case == "DIURNAL":
            dateString = '2022-06-21_2022-06-24'
    elif ModelData_NSSL.region == "Hawaii":
        if ModelData_NSSL.case == "TRADES":
            dateString = "2022-08-07_2022-08-10"
else:
    dateString = f"{ModelData_NSSL.simulationDates[0]}_{ModelData_NSSL.simulationDates[-1]}"

RadarData_MRMS = RadarData_MRMS_Class(ModelData_NSSL,
                                      fileDirectory=os.path.join(DirectoryManager.dataDirectory,
                                                                 f"Observation_Data/{ModelData_NSSL.region}/MRMS_RadarData",
                                                                 dateString,
                                                                 "MergedReflectivityQC_01.00"))

In [15]:
##########################
#DATA LOADING FUNCTIONS

In [16]:
#Converting timeStrings
def ConvertTimeStringtoDateTime(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    """
    return datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
def ConvertTimeStringtoTimeTitle(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    Formatted for use as a plot title.
    """
    # Parse the custom format to a datetime object
    dt = datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
    # Format it to 'YYYY-MM-DD HH:MM:SS'
    return dt.strftime('%Y-%m-%d %H:%M:%S')

In [17]:
def GetData(t):
    timeString = ModelData_NSSL.timeStrings[t]
    timeString_datetime = ConvertTimeStringtoDateTime(timeString)
    
    #Loading Observational Radar
    radarData, nearestFilePath = RadarData_MRMS.LoadClosestMRMSFile(target_time=timeString_datetime)
    radarData=radarData.isel(time=0)

    return radarData

In [18]:
def FixLatLon_RadarData(radarData):        
    radarData = radarData.isel(latitude=slice(None, None, -1))

    radarData = radarData.assign_coords(
        longitude=((radarData.longitude + 180) % 360) - 180
    )
    return radarData

def ReturnLatLon_RadarData(radarData):
    # Fix latitude order
    radarData_fixed = radarData.isel(latitude=slice(None, None, -1))

    # Fix longitude convention
    radarData_fixed = radarData_fixed.assign_coords(
        longitude=radarData.longitude+360
    )

    return radarData_fixed


def InterpolateRadarData(radarData,ModelData):
    radarData = FixLatLon_RadarData(radarData)
    
    radarData_interp = radarData.interp(
        latitude=ModelData.latitude,
        longitude=ModelData.longitude,
        method="linear"
    )
    return radarData_interp

In [19]:
def SaveMaskData(ModelData, RadarDataMask):
    outputPath = os.path.join(outputDirectory,f"{ModelData.region}_{ModelData.case}_spinup{ModelData.spinup_hours}hrs")
    os.makedirs(outputPath, exist_ok=True)
    outputFileName = f"RadarObservationMask.nc"
    outputFileNamePath = os.path.join(outputPath, outputFileName)

    RadarDataMask.to_netcdf(outputFileNamePath)
    print(f"Saved to {outputFileNamePath}","\n")

def SaveRadarObservationLevels(ModelData, RadarObservationLevels):
    outputPath = os.path.join(
        outputDirectory,
        f"{ModelData.region}_{ModelData.case}_spinup{ModelData.spinup_hours}hrs"
    )
    os.makedirs(outputPath, exist_ok=True)

    outputFileName = "RadarObservationLevels.pkl"
    outputFileNamePath = os.path.join(outputPath, outputFileName)

    with open(outputFileNamePath, "wb") as file:
        pickle.dump(RadarObservationLevels, file)

    print(f"Saved to {outputFileNamePath}\n")

In [20]:
##########################
#RUNNING

In [21]:

# Loading Data
radarData_t = GetData(t=0)
radarData_t_interp = InterpolateRadarData(radarData=radarData_t, ModelData=ModelData_NSSL)

#Creating Mask
RadarDataMask = xr.where(radarData_t_interp != -999, True, False)

#Saving Mask
SaveMaskData(ModelData_NSSL, RadarDataMask)

Target time:  2022-08-08 00:00:00
Closest file: MRMSReflectivity_HAWAII_Hawaii_20220808-000058.nc (2022-08-08 00:00:58)
Saved to /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/Observation_Data/RadarData/RadarObservationMask/Hawaii_TRADES_spinup0hrs/RadarObservationMask.nc 



In [22]:
RadarObservationLevels = [0.5, 0.75, 1.0, 
 1.25, 1.5, 1.75, 2.0,
 2.25, 2.5, 2.75, 3.0,
 3.5, 4.0,
 4.5, 5.0,
 5.5, 6.0,
 6.5, 7.0,
 7.5, 8.0,
 8.5, 9.0,
 10.0, 11.0,
 12.0, 13.0,
 14.0, 15.0,
 16.0, 17.0,
 18.0, 19.0]

SaveRadarObservationLevels(ModelData_NSSL, RadarObservationLevels)

#loading back in
# RadarObservationLevels = RadarObservationMask_Class.LoadRadarObservationLevels_MRMS(DirectoryManager, ModelData_NSSL)

Saved to /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/Observation_Data/RadarData/RadarObservationMask/Hawaii_TRADES_spinup0hrs/RadarObservationLevels.pkl

